# Gold-Patch Calibration

Runs Phase-2 verification with each task's **reference patch** in the same `subprocess` sandbox the dev-eval notebook uses. No LLM, CPU only. Any task whose gold patch fails here cannot be measured locally → listed in `calibration/env_broken.json`, excluded from the dev subset and bucketed as `env_broken` by `eval/triage.py`.

## 1. Environment

In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
# --- extra test deps + src-layout fix for the subprocess sandbox --------------
# In `sandbox="subprocess"` mode the harness never installs anything from
# EvalConfig.wheels_dir (install_test_dependencies / install_editable_package
# return early): tests run in a per-task venv that sees THIS kernel's
# site-packages, with PYTHONPATH=<sandbox>/workspace. So:
#  1. test deps missing from the Kaggle image (inline_snapshot, dirty_equals,
#     pytest-httpbin, trustme) are pip-installed into the kernel from the
#     tianyuxu2/gemma-4-extra-test-wheels dataset (already-satisfied deps are kept);
#  2. src-layout repos (psf/requests) would import the kernel's installed
#     `requests`, not the patched workspace copy; a .pth hook puts
#     <workspace>/src first on sys.path, but only inside swegemma sandboxes.
import site
import subprocess
import sys
from pathlib import Path

EXTRA_WHEELS_DIR = next(Path('/kaggle/input').rglob('inline_snapshot-*.whl')).parent
pip_freeze = lambda: set(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.split())
before = pip_freeze()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(EXTRA_WHEELS_DIR),
     'inline-snapshot', 'dirty-equals', 'pytest-httpbin', 'trustme'],
    check=True,
)
after = pip_freeze()
print('extra test deps installed:', sorted(after - before))
print('replaced:', sorted(before - after))

SRC_LAYOUT_HOOK = (
    "import os, sys; _s = [os.path.join(p, 'src') for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) "
    "if 'swegemma_sandbox_' in p and os.path.isdir(os.path.join(p, 'src'))]; "
    "sys.path[:0] = [p for p in _s if p not in sys.path]\n"
)
for sp in site.getsitepackages():
    if Path(sp).is_dir():
        (Path(sp) / '00_swegemma_src_layout.pth').write_text(SRC_LAYOUT_HOOK)
        print('src-layout hook ->', sp)
        break

# self-check in a system-site venv like the harness's: deps importable, src/ first
import os, tempfile, venv
_chk = Path(tempfile.mkdtemp(prefix='swegemma_sandbox_selfcheck_'))
(_chk / 'workspace' / 'src').mkdir(parents=True)
venv.create(_chk / 'venv', system_site_packages=True, with_pip=False, symlinks=True)
_res = subprocess.run(
    [str(_chk / 'venv' / 'bin' / 'python3'), '-c',
     'import sys, inline_snapshot, dirty_equals, pytest_httpbin, trustme; print(sys.path)'],
    env={**os.environ, 'PYTHONPATH': str(_chk / 'workspace')}, capture_output=True, text=True,
)
_ok = _res.returncode == 0 and str(_chk / 'workspace' / 'src') in _res.stdout
print('sandbox self-check:', 'OK' if _ok else f'FAILED\n{_res.stdout}\n{_res.stderr[-2000:]}')

## 2. Verify every task with its gold patch

In [ ]:
import asyncio
import json
import re
import time
from collections import Counter
from pathlib import Path

from swegemma.config import EvalConfig
from swegemma.deduplication import resolve_task_snapshot_paths
from swegemma.evaluate import Evaluator
from swegemma.harness.verification import verify_task
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
OUT_DIR = WORKING_DIR / 'calibration'
(OUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)
CONCURRENCY = 3  # CPU notebook has 4 cores; keep one free so timeouts are not load-induced

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
print(f'Loaded {len(tasks)} tasks')

# Same subprocess sandbox + verification path as the dev-eval notebook, but no LLM:
# the gold `patch` is passed in as the agent patch.
config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=OUT_DIR,
    submission_dir=WORKING_DIR,
    models=None,
    sandbox='subprocess',
    timeout_seconds=300,
    verification_timeout_seconds=300,
    wheels_dir=DATA_DIR / 'wheels',
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    display_mode='quiet',
)
evaluator = Evaluator(config)

# Signatures of a broken local test environment (vs. a genuinely failing test)
ENV_PATTERNS = {
    'missing_module': r"ModuleNotFoundError: No module named '([^']+)'",
    'fixture_recursion': r"recursive dependency involving fixture '([^']+)'",
    'fixture_missing': r"fixture '([^']+)' not found",
    'import_error': r'ImportError(?: while importing test module)?:? (.+)',
}


def env_reasons(text):
    found = []
    for kind, pat in ENV_PATTERNS.items():
        for m in dict.fromkeys(re.findall(pat, text or '')):
            found.append(f'{kind}:{m}'[:160])
    return found


async def check(task, sem):
    async with sem:
        t0 = time.perf_counter()
        snap, base_snap, patch_path = resolve_task_snapshot_paths(config.snapshots_dir, task.instance_id, task.repo)
        try:
            res = await verify_task(
                evaluator.docker, config, task, snap,
                base_snapshot_path=base_snap, patch_path=patch_path,
                agent_patch=task.patch, start_time=t0,
            )
            out, ec, err, ok = res.test_output or '', res.test_exit_code, res.error or '', bool(res.resolved)
        except Exception as e:  # harness crash counts as environment breakage
            out, ec, err, ok = '', -1, f'{type(e).__name__}: {e}', False
        (OUT_DIR / 'logs' / f'{task.instance_id}.log').write_text(out, encoding='utf-8')
        row = {
            'id': task.instance_id, 'repo': task.repo, 'gold_resolved': ok,
            'test_exit_code': ec, 'error': err[:500],
            'env_reasons': [] if ok else env_reasons(out + '\n' + err),
            'duration_seconds': round(time.perf_counter() - t0, 1),
        }
        print(f"{'OK  ' if ok else 'FAIL'} {task.instance_id:18} ec={ec} {row['duration_seconds']:6.1f}s {row['env_reasons'][:2]}", flush=True)
        return row


async def main():
    sem = asyncio.Semaphore(CONCURRENCY)
    return await asyncio.gather(*(check(t, sem) for t in tasks))


# Notebook kernels already run an event loop; run the sweep in a worker thread then.
try:
    asyncio.get_running_loop()
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        rows = pool.submit(lambda: asyncio.run(main())).result()
except RuntimeError:
    rows = asyncio.run(main())

rows.sort(key=lambda r: r['id'])
with (OUT_DIR / 'gold_results.jsonl').open('w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')
env_broken = sorted(r['id'] for r in rows if not r['gold_resolved'])
(OUT_DIR / 'env_broken.json').write_text(json.dumps(env_broken, indent=1) + '\n', encoding='utf-8')

print(f"\ngold resolved: {len(rows) - len(env_broken)}/{len(rows)}")
for repo, n in sorted(Counter(r['repo'] for r in rows if not r['gold_resolved']).items()):
    tot = sum(r['repo'] == repo for r in rows)
    print(f'  {repo}: {n}/{tot} env-broken')
print('reasons:', Counter(x.split(':')[0] + ':' + x.split(':')[1] for r in rows for x in r['env_reasons']).most_common(15))
print('wrote', OUT_DIR / 'env_broken.json')
